# 00 — Provenance : comment `eli_lilly.csv` a été extrait d'Open Medic

Ce notebook documente et **reproduit** l'extraction des quatre produits Eli Lilly
depuis les fichiers bruts Open Medic (AMELI), pour que le pipeline complet soit
traçable — pas seulement le fichier déjà extrait.

**Il ne modifie pas `data/eli_lilly.csv`** (versionné dans le repo, utilisé par
tous les autres notebooks). Il ré-extrait dans `outputs/` et **compare** au
fichier de référence, pour prouver que l'extraction est reproductible.

**Si vous n'avez pas les fichiers bruts Open Medic sur votre machine**, ce
notebook s'arrête proprement après la cellule 1 — `data/eli_lilly.csv` reste
utilisable tel quel pour tous les autres notebooks.


## 0. Recherche des fichiers bruts

In [1]:
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUTPUTS_DIR = PROJECT_DIR / 'outputs'
OUTPUTS_DIR.mkdir(exist_ok=True)

CANDIDATS_RAW = [
    PROJECT_DIR / 'data' / 'raw',
    Path('C:/Users/Enes/data'),
]
RAW_DIR = next((p for p in CANDIDATS_RAW if p.exists() and any(p.glob('OPEN_MEDIC_*'))), None)

if RAW_DIR is None:
    print("Aucun fichier Open Medic brut trouvé — ce notebook s'arrête ici.")
    print("data/eli_lilly.csv (déjà extrait) reste utilisable normalement.")
else:
    print('Fichiers bruts trouvés dans :', RAW_DIR)

Fichiers bruts trouvés dans : C:\Users\Enes\data


---
## 1. Méthode d'extraction

Les quatre produits Lilly sont repérés par un **mot-clé dans le nom commercial**
(`l_cip13`), qui identifie sans ambiguïté chaque médicament quel que soit son
dosage ou sa forme galénique (ex. `TRULICITY 0,75 MG` et `TRULICITY 1,5 MG`
sont tous deux capturés par le mot-clé `TRULICITY`).

In [2]:
LILLY_MAP = {
    'TRULICITY': ('Trulicity', 'Dulaglutide',   'Diabète'),
    'VERZENIO':  ('Verzenio',  'Abemaciclib',   'Oncologie'),
    'JARDIANCE': ('Jardiance', 'Empagliflozin', 'Diabète / Cardiovasculaire'),
    'CYMBALTA':  ('Cymbalta',  'Duloxétine',    'Antidépresseur'),
}

AGE_LABELS  = {0: '<20 ans', 20: '20-59 ans', 60: '60+ ans', 99: 'Inconnu'}
SEXE_LABELS = {1: 'Homme', 2: 'Femme', 9: 'Inconnu'}
REG_LABELS  = {
    1: 'Guadeloupe', 2: 'Martinique', 3: 'Guyane', 4: 'La Réunion', 5: 'Mayotte',
    11: 'Île-de-France', 24: 'Centre-Val de Loire', 27: 'Bourgogne-Franche-Comté',
    28: 'Normandie', 32: 'Hauts-de-France', 44: 'Grand Est',
    52: 'Pays de la Loire', 53: 'Bretagne', 75: 'Nouvelle-Aquitaine',
    76: 'Occitanie', 84: 'Auvergne-Rhône-Alpes', 93: 'PACA', 94: 'Corse',
    99: 'Inconnu / Étranger',
}

def parse_euro(series):
    return (series.astype(str).str.strip()
            .str.replace('.', '', regex=False)
            .str.replace(',', '.', regex=False)
            .replace('', '0').astype(float))

def extraire_annee(annee, raw_dir):
    """Charge une année Open Medic et ne garde que les lignes Lilly."""
    for pattern in [f'OPEN_MEDIC_{annee}.zip', f'OPEN_MEDIC_{annee}.csv']:
        path = raw_dir / pattern
        if path.exists():
            break
    else:
        return None

    # Séparateur ';' explicite : Open Medic est toujours au même format,
    # le sniffer sep=None est ~3x plus lent pour un résultat identique.
    df = pd.read_csv(path, sep=';', encoding='latin-1')
    df.columns = df.columns.str.lower().str.strip()
    df['rem'] = parse_euro(df['rem'])

    frames = []
    for mot_cle, (nom, molecule, categorie) in LILLY_MAP.items():
        subset = df.loc[df['l_cip13'].str.upper().str.contains(mot_cle, na=False)].copy()
        if subset.empty:
            continue
        subset['nom_lilly'] = nom
        subset['molecule']  = molecule
        subset['categorie'] = categorie
        frames.append(subset)

    if not frames:
        return None
    out = pd.concat(frames, ignore_index=True)
    out['annee'] = annee
    return out

## 2. Extraction sur toute la période (si les fichiers bruts sont disponibles)

In [3]:
if RAW_DIR is not None:
    frames = []
    for annee in range(2016, 2026):
        agg = extraire_annee(annee, RAW_DIR)
        if agg is not None:
            frames.append(agg)
            print(f'[{annee}] {agg.shape[0]:,} lignes Lilly extraites')

    df_reextrait = pd.concat(frames, ignore_index=True)
    df_reextrait['age_label']  = df_reextrait['age'].map(AGE_LABELS)
    df_reextrait['sexe_label'] = df_reextrait['sexe'].map(SEXE_LABELS)
    df_reextrait['reg_label']  = df_reextrait['ben_reg'].map(REG_LABELS)

    print(f'\nTotal ré-extrait : {df_reextrait.shape[0]:,} lignes')

[2016] 1,255 lignes Lilly extraites
[2017] 1,198 lignes Lilly extraites
[2018] 1,190 lignes Lilly extraites
[2019] 1,327 lignes Lilly extraites
[2020] 1,283 lignes Lilly extraites
[2021] 1,921 lignes Lilly extraites
[2022] 2,367 lignes Lilly extraites
[2023] 2,820 lignes Lilly extraites
[2024] 3,088 lignes Lilly extraites
[2025] 3,681 lignes Lilly extraites

Total ré-extrait : 20,130 lignes


---
## 3. Validation : la ré-extraction reproduit-elle `data/eli_lilly.csv` ?

On compare les remboursements totaux par produit et par année entre le fichier
de référence (versionné) et la ré-extraction depuis les données brutes.

In [4]:
if RAW_DIR is not None:
    reference = pd.read_csv(PROJECT_DIR / 'data' / 'eli_lilly.csv')

    ref_check    = reference.groupby(['annee', 'nom_lilly'])['rem'].sum().round(0)
    reextr_check = df_reextrait.groupby(['annee', 'nom_lilly'])['rem'].sum().round(0)

    comparaison = pd.DataFrame({
        'reference_eur':    ref_check,
        'reextraction_eur': reextr_check,
    })
    comparaison['ecart_eur'] = (comparaison['reextraction_eur'] - comparaison['reference_eur']).fillna(0)

    n_lignes_ref = len(reference)
    n_lignes_reextr = len(df_reextrait)
    ecart_max = comparaison['ecart_eur'].abs().max()

    print(f'Lignes  — référence : {n_lignes_ref:,}  |  ré-extraction : {n_lignes_reextr:,}')
    print(f'Écart maximal sur un couple (année, produit) : {ecart_max:.2f} €')
    print('\n✅ Extraction reproductible' if ecart_max < 1 else '\n⚠️ Écart détecté — à investiguer')

    # Sauvegarde de la ré-extraction pour archive (ne remplace jamais data/eli_lilly.csv)
    export_path = OUTPUTS_DIR / 'eli_lilly_reextrait.csv'
    df_reextrait.to_csv(export_path, index=False, encoding='utf-8')
    print(f'\nRé-extraction archivée : {export_path}')

    comparaison

Lignes  — référence : 20,130  |  ré-extraction : 20,130
Écart maximal sur un couple (année, produit) : 0.00 €

✅ Extraction reproductible

Ré-extraction archivée : c:\Users\Enes\Desktop\projet_lilly\outputs\eli_lilly_reextrait.csv
